<a href="https://colab.research.google.com/github/ThayseBachmeyer/ProcessamentoSinaisI/blob/main/Aula%2004/C%C3%B3digos/Prat4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Código utilizado na questão 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1.0, -1.0, 1.0, -1.0])

valN = [4, 16, 64, 1024]

def dtft(x,omega):
  n = np.arange(len(x))
  return np.array([np.sum(x * np.exp(-1j * w * n)) for w in omega])

def m_dft(N):
  k = np.arange(N).reshape((N, 1))
  n = np.arange(N).reshape((1, N))
  return np.exp(-2j * np.pi * k * n / N)

def calc_dft(x, N):
  x_pad = np.zeros(N, dtype = complex)
  x_pad[:len(x)] = x
  return m_dft(N) @ x_pad

omega_dtft = np.linspace(0, 2 * np.pi, 2000, endpoint = False)
x_dtft = dtft(x, omega_dtft)

fig, axs = plt.subplots(len(valN), 2, figsize=(10,12))

for i, N in enumerate(valN):
  x_dft = calc_dft(x, N)
  omega_dft = 2 * np.pi * np.arange(N) / N

  ax_m = axs[i, 0]
  ax_m.plot(omega_dtft / np.pi, np.abs(x_dtft), label = 'dtft', color = "black")
  ax_m.plot(omega_dft / np.pi, np.abs(x_dft), label = 'dft', color = "red")
  ax_m.set_title(f'Magnitude para N = {N}')
  ax_m.set_xlabel('Frequência (π)')
  ax_m.set_ylabel('Magnitude')
  ax_m.grid(alpha = 0.3)
  if i == 0:
    ax_m.legend(['dtft', 'dft'])

  ax_f = axs[i, 1]
  ax_f.plot(omega_dtft / np.pi, np.unwrap(np.angle(x_dtft)), label = 'dtft', color = "black")
  ax_f.plot(omega_dft / np.pi, np.unwrap(np.angle(x_dft)), label = 'dft', color = "red")
  ax_f.set_title(f'Fase para N = {N}')
  ax_f.set_xlabel('Frequência (π)')
  ax_f.set_ylabel('Fase (rad)')
  ax_f.grid(alpha = 0.3)

plt.tight_layout()
plt.show()

## Código utilizado na questão 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1.0, -1.0, 1.0, -1.0])

def sinal(n_a):
  n = np.arange(n_a)
  t = n / fs
  return np.sin(2 * np.pi * t) + np.sin(2.2 * np.pi * t)
fs = 10

def m_dft(N):
  k = np.arange(N).reshape((N, 1))
  n = np.arange(N).reshape((1, N))
  return np.exp(-2j * np.pi * k * n / N)

def c_dft(x, N):
  x_pad = np.zeros(N, dtype = complex)
  x_pad[:len(x)] = x
  return m_dft(N) @ x_pad

x64 = sinal(64)
x128 = sinal(128)

fig, axs = plt.subplots(1, 2, figsize = (10,5))

for ax, (nome, N) in zip(axs, [('N = 64', 64), ('N = 128', 128)]):
  X = c_dft (x64, N)
  freqs = np.arange(N) * fs / N
  ax.plot(freqs, np.abs(X), 'o-', markersize = 2)
  ax.set_xlim(0, 2.5)
  ax.set_title(nome)
  ax.set_xlabel('Frequência (Hz)')
  ax.set_ylabel('Magnitude')
  ax.axvline(1.0, color = 'gray', ls = ':', lw = 1)
  ax.axvline(1.1, color = 'gray', ls = ':', lw = 1)
  ax.grid(alpha = 0.3)

plt.tight_layout()
plt.show()

fig, axs = plt.subplots(1, 3, figsize = (13, 5))

for ax, (nome, N) in zip(axs, [('N = 128', 128), ('N = 256)', 256), ('N = 512', 512)]):
  X = c_dft (x128, N)
  freqs = np.arange(N) * fs / N
  ax.plot(freqs, np.abs(X), 'o-', markersize = 2)
  ax.set_xlim(0, 2.5)
  ax.set_title(nome)
  ax.set_xlabel('Frequência (Hz)')
  ax.set_ylabel('Magnitude')
  ax.axvline(1.0, color = 'gray', ls = ':', lw = 1)
  ax.axvline(1.1, color = 'gray', ls = ':', lw = 1)
  ax.grid(alpha = 0.3)

plt.tight_layout()
plt.show

## Código utilizado na questão 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from IPython.display import Audio, display

handel = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2003/Dados/handel.wav"
nome_handel = "handel.wav"

!wget -q -O {nome_handel} {handel}

fs, x = wavfile.read(nome_handel)
if len(x.shape) > 1:
    x = x[:, 0]

x = x.astype(np.float64)
x = x / np.max(np.abs(x))
Nt = len(x)
t = np.arange(Nt) / fs

def m_dft(N):
  k = np.arange(N).reshape((N, 1))
  n = np.arange(N).reshape((1, N))
  return np.exp(-2j * np.pi * k * n / N)

tam_bloco = 1024
n_blocos = int(np.ceil(Nt / tam_bloco))
Npad = n_blocos * tam_bloco
xpad = np.zeros(Npad)
xpad[:Nt] = x
blocos = xpad.reshape(n_blocos, tam_bloco)

W = m_dft(tam_bloco)
Winv = np.conj(W).T / tam_bloco
Xbloco = blocos @ W.T

meio = tam_bloco // 2
peso = np.abs(Xbloco[:, :meio+1])**2
peso[:, 1:meio] *= 2
peso_flat = peso.flatten()
energiaT = np.sum(np.abs(Xbloco)**2)
ordem = np.argsort(peso_flat)[::-1]
csum = np.cumsum(peso_flat[ordem])

valr = [0.995,  0.99, 0.9, 0.75, 0.5]
resultados = {}

for r in valr:
  n_sel = np.searchsorted(csum, r *  energiaT) + 1
  mask_meio = np.zeros_like(peso_flat, dtype = bool)
  mask_meio[ordem[:n_sel]] = True
  mask_meio = mask_meio.reshape(n_blocos, meio+1)

  mask = np.zeros((n_blocos, tam_bloco), dtype=bool)
  mask[:, :meio+1] = mask_meio
  mask[:, meio+1:] = mask_meio[:, 1:meio][:, ::-1]

  X_comp = Xbloco * mask
  x_rec = (X_comp @ Winv.T).real.flatten()[:Nt]
  mse = np.mean((x - x_rec)**2)
  resultados[r] = dict(x_rec=x_rec, n_sel=n_sel, mse=mse)

  total_unicos = n_blocos * (meio+1)
  print(f'r={r*100:5.1f}%  coeficientes únicos={n_sel:6d} de {total_unicos} 'f'({100*n_sel/total_unicos:5.2f}%)  MSE={mse:.3e}')

rs = np.array(valr)*100
mses = [resultados[r]['mse'] for r in valr]
ncoefs = [resultados[r]['n_sel'] for r in valr]

fig1, ax1 = plt.subplots(1, 2, figsize=(11, 4))
ax1[0].semilogy(rs, mses, 'o-')
ax1[0].set_xlabel('r (% de energia retida)'); ax1[0].set_ylabel('MSE')
ax1[0].set_title('Erro quadrático médio vs r')
ax1[0].grid(alpha=0.3, which='both')

ax1[1].plot(rs, ncoefs, 'o-', color='tab:orange')
ax1[1].set_xlabel('r (% de energia retida)'); ax1[1].set_ylabel('N° de coeficientes únicos')
ax1[1].set_title('Coeficientes necessários vs r')
ax1[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

janela = slice(int(2.0*fs), int(2.05*fs))
fig2, eixos2 = plt.subplots(len(valr), 1, figsize=(9, 10), sharex=True)
for ax, r in zip(eixos2, valr):
    ax.plot(t[janela], x[janela], label='original', linewidth=1)
    ax.plot(t[janela], resultados[r]['x_rec'][janela], '--', label=f'r={r*100:.1f}%', linewidth=1)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(alpha=0.3)
eixos2[-1].set_xlabel('Tempo (s)')
plt.suptitle('Sinal original vs. comprimido (zoom de 50 ms)')
plt.tight_layout()
plt.show()

print('Original:'); display(Audio(x, rate=fs))
for r in valr:
    print(f'Comprimido, r={r*100:.1f}%:')
    display(Audio(resultados[r]['x_rec'], rate=fs))